# 🔵 XLM-RoBERTa — Multilabel Complaint Classification
### Thesis Project | Fixed & Complete Version
---
**Labels:** Billing, Refund, Delivery, Customer_Service, Product_Quality, Account, Technical, Network

## Step 1 — Install Libraries

In [ ]:
!pip install transformers scikit-learn openpyxl torch -q

##Step 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, hamming_loss

print(' Libraries imported')
print(' Device:', 'GPU' if torch.cuda.is_available() else 'CPU  (GPU recommend)')

✅ Libraries imported
🖥️ Device: GPU ✅


##  Step 3 — Config

In [ ]:

TRAIN_FILE = '/content/Final_Cleaned_Labels.xlsx'
TEST_FILE  = '/content/Processed_Data.xlsx'

MODEL_NAME  = 'xlm-roberta-base'
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 5
LR          = 2e-5
THRESHOLD   = 0.5
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABELS = ['Billing', 'Refund', 'Delivery', 'Customer_Service',
          'Product_Quality', 'Account', 'Technical', 'Network']

print('✅ Config set')
print(f'   Labels ({len(LABELS)}):', LABELS)

✅ Config set
   Labels (8): ['Billing', 'Refund', 'Delivery', 'Customer_Service', 'Product_Quality', 'Account', 'Technical', 'Network']


##Step 4 — Upload Train & Test Files

In [ ]:
from google.colab import files
print(' Step 1: Final_Cleaned_Labels.xlsx upload করো (train data)')
uploaded = files.upload()
print('\n  Step 2: Processed_Data.xlsx upload করো (test data)')
uploaded = files.upload()

📂 Step 1: Final_Cleaned_Labels.xlsx upload করো (train data)


Saving Final_Cleaned_Labels.xlsx to Final_Cleaned_Labels.xlsx

📂 Step 2: Processed_Data.xlsx upload করো (test data)


Saving Processed_Data.xlsx to Processed_Data.xlsx


##Step 5 — Load & Binarize Data

In [ ]:
mlb = MultiLabelBinarizer(classes=LABELS)

def load_and_binarize(file_path, fit=False, text_col='text', label_col='labels'):
    df = pd.read_excel(file_path).dropna(subset=[label_col])
    df[label_col] = df[label_col].str.strip()

    df['labels_list'] = df[label_col].apply(lambda x: [l.strip().title() for l in x.split(';') if l.strip()])

    y = mlb.fit_transform(df['labels_list']) if fit else mlb.transform(df['labels_list'])
    return df[text_col].values, y

X_train, y_train = load_and_binarize(TRAIN_FILE, fit=True, text_col='text', label_col='labels')   # fit=True শুধু train এ
X_test,  y_test  = load_and_binarize(TEST_FILE,  fit=False, text_col='Complaint', label_col='Labels')  # test এ শুধু transform

print(f' Train: {len(X_train)} samples')
print(f' Test:  {len(X_test)} samples')
print(f' Labels: {mlb.classes_}')
print(f' y_train shape: {y_train.shape}')

✅ Train: 3278 samples
✅ Test:  795 samples
✅ Labels: ['Billing' 'Refund' 'Delivery' 'Customer_Service' 'Product_Quality'
 'Account' 'Technical' 'Network']
✅ y_train shape: (3278, 8)


In [ ]:
import pandas as pd

# Load the test file to inspect its columns
df_test_columns = pd.read_excel(TEST_FILE)
print("Columns in Processed_Data.xlsx:")
print(df_test_columns.columns.tolist())

# Please update the 'label_col' in the cell above (cell-load) with the correct column name found here.

Columns in Processed_Data.xlsx:
['Complaint', 'Labels']


##Step 6 — Tokenizer & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ComplaintDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_loader = DataLoader(ComplaintDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(ComplaintDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

print(' Tokenizer loaded')
print(f' Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

✅ Tokenizer loaded
✅ Train batches: 205 | Test batches: 50


##Step 7 — Load XLM-RoBERTa Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type='multi_label_classification'
).to(DEVICE)

print('✅ XLM-RoBERTa model loaded')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'   Device: {DEVICE}')

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ XLM-RoBERTa model loaded
   Parameters: 278,049,800
   Device: cuda


##  Step 8 — Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.BCEWithLogitsLoss()

print('Training Started...')
print('=' * 45)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=ids, attention_mask=mask)
        loss    = criterion(outputs.logits, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

print('=' * 45)
print('Training End')

🚀 Training শুরু হচ্ছে...
Epoch 1/5  |  Loss: 0.4383
Epoch 2/5  |  Loss: 0.3081
Epoch 3/5  |  Loss: 0.2343
Epoch 4/5  |  Loss: 0.1899
Epoch 5/5  |  Loss: 0.1615
✅ Training শেষ!


## Step 9 — Evaluation (Thesis Result)

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        logits = model(input_ids=ids, attention_mask=mask).logits
        preds  = (torch.sigmoid(logits) > THRESHOLD).cpu().numpy()
        all_preds.append(preds)
        all_true.append(batch['labels'].numpy())

all_preds = np.vstack(all_preds)
all_true  = np.vstack(all_true)

print('=' * 55)
print('       XLM-RoBERTa — Evaluation Results')
print('=' * 55)
print(classification_report(all_true, all_preds, target_names=LABELS, zero_division=0))
print(f'Hamming Loss : {hamming_loss(all_true, all_preds):.4f}')
print(f'F1 Micro     : {f1_score(all_true, all_preds, average="micro", zero_division=0):.4f}')
print(f'F1 Macro     : {f1_score(all_true, all_preds, average="macro", zero_division=0):.4f}')
print(f'F1 Weighted  : {f1_score(all_true, all_preds, average="weighted", zero_division=0):.4f}')

       XLM-RoBERTa — Evaluation Results
                  precision    recall  f1-score   support

         Billing       0.90      0.38      0.53       119
          Refund       0.82      0.88      0.85        91
        Delivery       0.83      0.72      0.77       117
Customer_Service       0.79      0.71      0.75       126
 Product_Quality       0.69      0.83      0.75       100
         Account       0.84      0.76      0.80       161
       Technical       0.47      0.75      0.58       108
         Network       0.80      0.89      0.84        79

       micro avg       0.74      0.73      0.73       901
       macro avg       0.77      0.74      0.73       901
    weighted avg       0.77      0.73      0.73       901
     samples avg       0.75      0.75      0.73       901

Hamming Loss : 0.0753
F1 Micro     : 0.7326
F1 Macro     : 0.7340
F1 Weighted  : 0.7304


In [ ]:
import numpy as np
import torch
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        logits = model(input_ids=ids, attention_mask=mask).logits
        preds  = (torch.sigmoid(logits) > THRESHOLD).cpu().numpy()
        all_preds.append(preds)
        all_true.append(batch['labels'].numpy())

all_preds = np.vstack(all_preds)
all_true  = np.vstack(all_true)

exact_match_accuracy = np.all(all_true == all_preds, axis=1).mean()
print(f'Exact Match Accuracy: {exact_match_accuracy:.4f}')

Exact Match Accuracy: 0.6214


##Step 10 — Save Model

In [ ]:
model.save_pretrained('./xlmr_final')
tokenizer.save_pretrained('./xlmr_final')
print('Model saved → ./xlmr_final/')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved → ./xlmr_final/


##Step 11 — Predict Function

In [ ]:
def predict(text, threshold=THRESHOLD):
    model.eval()
    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=mask).logits
        probs  = torch.sigmoid(logits).cpu().numpy()[0]

    results = [(LABELS[i], round(float(probs[i]), 3)) for i in range(len(LABELS)) if probs[i] > threshold]
    if not results:
        best = int(np.argmax(probs))
        results = [(LABELS[best], round(float(probs[best]), 3))]

    return results

test_texts = [
    'আমার অর্ডার ডেলিভারি হয়নি এবং টাকাও ফেরত পাইনি',
    'কাস্টমার সার্ভিস একদম বাজে, কেউ ফোন ধরে না',
    'নেটওয়ার্ক সমস্যার কারণে পেমেন্ট হয়নি',
    'লগইন করতে পারতেছি না অ্যাকাউন্টে ঢুকতে পারছি না',
]

print('=' * 50)
print('  Sample Predictions')
print('=' * 50)
for t in test_texts:
    print(f'\n📝 {t}')
    print(f'🏷️  {predict(t)}')

  Sample Predictions

📝 আমার অর্ডার ডেলিভারি হয়নি এবং টাকাও ফেরত পাইনি
🏷️  [('Refund', 0.876), ('Delivery', 0.851)]

📝 কাস্টমার সার্ভিস একদম বাজে, কেউ ফোন ধরে না
🏷️  [('Customer_Service', 0.962)]

📝 নেটওয়ার্ক সমস্যার কারণে পেমেন্ট হয়নি
🏷️  [('Network', 0.968)]

📝 লগইন করতে পারতেছি না অ্যাকাউন্টে ঢুকতে পারছি না
🏷️  [('Account', 0.97)]


In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             hamming_loss, matthews_corrcoef,
                             multilabel_confusion_matrix,
                             classification_report)
import numpy as np

# Re-run evaluation
model.eval()
all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        logits = model(input_ids=ids, attention_mask=mask).logits
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs > THRESHOLD)
        all_preds.append(preds)
        all_true.append(batch['labels'].numpy())
        all_probs.append(probs)

all_preds = np.vstack(all_preds)
all_true  = np.vstack(all_true)
all_probs = np.vstack(all_probs)

print('=' * 55)
print('        XLM-RoBERTa — Complete Evaluation')
print('=' * 55)

# Per-label classification report
print('\n📊 Per-Label Results:')
print(classification_report(all_true, all_preds, target_names=LABELS, zero_division=0))

# Overall metrics
print('\n Overall Metrics:')
print(f'  Accuracy (Exact Match) : {accuracy_score(all_true, all_preds):.4f}')
print(f'  F1 Micro               : {f1_score(all_true, all_preds, average="micro", zero_division=0):.4f}')
print(f'  F1 Macro               : {f1_score(all_true, all_preds, average="macro", zero_division=0):.4f}')
print(f'  F1 Weighted            : {f1_score(all_true, all_preds, average="weighted", zero_division=0):.4f}')
print(f'  Hamming Loss           : {hamming_loss(all_true, all_preds):.4f}')
print(f'  ROC-AUC (Macro)        : {roc_auc_score(all_true, all_probs, average="macro"):.4f}')

# Per-label MCC
print('\n Per-Label MCC:')
for i, label in enumerate(LABELS):
    mcc = matthews_corrcoef(all_true[:, i], all_preds[:, i])
    print(f'  {label:20s} : {mcc:.4f}')

# Confusion Matrix
print('\n Per-Label Confusion Matrix:')
print(f'  {"Label":20s}   TP    FP    FN    TN')
print('  ' + '-' * 45)
mcm = multilabel_confusion_matrix(all_true, all_preds)
for i, label in enumerate(LABELS):
    tn, fp, fn, tp = mcm[i].ravel()
    print(f'  {label:20s}  {tp:4d}  {fp:4d}  {fn:4d}  {tn:4d}')

        XLM-RoBERTa — Complete Evaluation

📊 Per-Label Results:
                  precision    recall  f1-score   support

         Billing       0.90      0.38      0.53       119
          Refund       0.82      0.88      0.85        91
        Delivery       0.83      0.72      0.77       117
Customer_Service       0.79      0.71      0.75       126
 Product_Quality       0.69      0.83      0.75       100
         Account       0.84      0.76      0.80       161
       Technical       0.47      0.75      0.58       108
         Network       0.80      0.89      0.84        79

       micro avg       0.74      0.73      0.73       901
       macro avg       0.77      0.74      0.73       901
    weighted avg       0.77      0.73      0.73       901
     samples avg       0.75      0.75      0.73       901


📊 Overall Metrics:
  Accuracy (Exact Match) : 0.6214
  F1 Micro               : 0.7326
  F1 Macro               : 0.7340
  F1 Weighted            : 0.7304
  Hamming Loss         